## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        # MNIST 输入是 28*28=784
        # 隐藏层 256, 输出层 10 (10分类)
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        # x shape: (batch_size, 28, 28) -> 平铺成 (batch_size, 784)
        x = tf.reshape(x, [-1, 784])
        # 第一层：Linear + ReLU
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        # 第二层：Linear 
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [10]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.49189392 ; accuracy 0.8638667
epoch 1 : loss 0.49164042 ; accuracy 0.86395
epoch 2 : loss 0.49138755 ; accuracy 0.86406666
epoch 3 : loss 0.49113518 ; accuracy 0.86406666
epoch 4 : loss 0.49088344 ; accuracy 0.8641
epoch 5 : loss 0.49063224 ; accuracy 0.86415
epoch 6 : loss 0.4903816 ; accuracy 0.8641667
epoch 7 : loss 0.4901316 ; accuracy 0.8641833
epoch 8 : loss 0.48988208 ; accuracy 0.8642
epoch 9 : loss 0.48963323 ; accuracy 0.8642833
epoch 10 : loss 0.4893849 ; accuracy 0.86431664
epoch 11 : loss 0.4891371 ; accuracy 0.86433333
epoch 12 : loss 0.48888987 ; accuracy 0.86438334
epoch 13 : loss 0.48864323 ; accuracy 0.86448336
epoch 14 : loss 0.48839712 ; accuracy 0.8645667
epoch 15 : loss 0.48815152 ; accuracy 0.8645833
epoch 16 : loss 0.48790646 ; accuracy 0.8645833
epoch 17 : loss 0.487662 ; accuracy 0.8647
epoch 18 : loss 0.487418 ; accuracy 0.86475
epoch 19 : loss 0.4871746 ; accuracy 0.86481667
epoch 20 : loss 0.48693174 ; accuracy 0.8648667
epoch 21 : loss 0.4